EDA - Datasets are extracted from: "[here](https://www.kaggle.com/datasets/nathansmallcalder/lol-match-history-and-summoner-data-80k-matches)"

Posibles hipótesis:
- La gente se toma más en serio las partidas ranked que las normales (comparar RankFk '0' con el resto)
- El oro total está más fuertemente asociado con la victoria que el KDA | La diferencia de oro entre equipos es mayor en partidas ganadas que la diferencia de KDA
- La diferencia de objetivos entre equipos es mayor que la diferencia en kills en partidas ganadas.


In [1]:
import os
import numpy as np
import pandas as pd
os.chdir("..")

from utils import clean_utils

Primero cargamos los 6 datasets

In [ ]:
# ID matching dataframes
df_champions = pd.read_csv("datasets/ChampionTbl.csv", sep=",")  # Links ChampionId (ChampionFk) and ChampionName
df_items = pd.read_csv("datasets/ItemTbl.csv", sep=",")  # Links ItemID and ItemName
df_rank = pd.read_csv("datasets/RankTbl.csv", sep=",")  # Links RankId (RankFk) and RankName 
df_summoner_match = pd.read_csv("datasets/SummonerMatchTbl.csv", sep=",")  # Links SummonerFk, MatchFk, and ChampionFk by SummonerMatchId (SummonerMatchFk)


# Actual info dataframes
df_match = pd.read_csv("datasets/MatchTbl.csv", sep=",")  # Contains general stats for any MatchFk (Patch, QueueType, RankFk, GameDuration)

df_match_stats = pd.read_csv("datasets/MatchStatsTbl.csv", sep=",")  # Personal match summoner stats for any SummonerMatchFk (MinionsKilled, DmgDealt, Win/Loss, ...)

df_team_match_stats = pd.read_csv("datasets/TeamMatchTbl.csv", sep=",")  # Team match stats for any TeamID (MatchFk, B1Champ, Win/Loss, ...)


# patch_info = clean_utils.parse_patches(df_match["Patch"])
# patch_info.value_counts().sort_values(ascending=False)

El análisis va a ser realizado sobre las partidas del modo clásico. Seleccionamos únicamente las partidas cuya "QueueType" sea clásico.

In [4]:
classic_mask = df_match["QueueType"] == "CLASSIC"
df_classic_match = df_match[classic_mask]
df_classic_match["RankFk"].value_counts()

RankFk
0     49483
8     32284
4     23249
5     21366
7     19588
6     17157
3     15529
2      6861
1      2425
9      2391
10      455
Name: count, dtype: int64

Una vez que tenemos las partidas clásicas solo, hacemos el merge para conseguir las estadísticas de las otras tablas.

In [5]:
total_data = pd.merge(df_classic_match, df_summoner_match, how="inner", left_on="MatchId", right_on="MatchFk")
total_data2 = pd.merge(total_data, df_match_stats, how="inner", left_on="SummonerMatchId", right_on="SummonerMatchFk")

total_data2["Lane"].value_counts()

Lane
BOTTOM     107992
JUNGLE      89818
MIDDLE      86549
TOP         81650
UTILITY     54149
NONE        23230
SUPPORT        12
Name: count, dtype: int64